# Create Wihuri Foundation Awards (Jenny ja Antti Wihurin rahasto)

Creates Jenny and Antti Wihuri Foundation awards from the foundation's own
public list of funded grants, https://wihurinrahasto.fi/myonnot/
("Rahoitetut hankkeet"): every general-call grant since 2014 with year,
grantee, purpose line and amount (EUR). Server-rendered HTML, 20 rows per
page, no export and no per-grant pages (the WP REST API is 401).
**2,759 science grants, 2014-2025, EUR 79.8M, 100% title/amount/year,
91% named lead** (local run 2026-10-01).

**Scope filter (stated per batch rule):** only the foundation's own
**science** category (`?myonto_kategoria=tiede`). Wihuri is a mixed
foundation; its **arts** (`taide`) and **societal-activity** (`yht`)
grants are excluded as non-research money. Inside the science category all
rows are kept, including 263 grants (9.5%) to organisations (learned
societies, research seminars, the mathematics journal Solmu, student
entrepreneurship societies); these are flagged by `grantee_type`.

**Prerequisites:**
- Run `scripts/local/wihuri_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/wihuri/wihuri_projects.parquet` (1.4 shrink guard).

**Field mapping:**
- `display_name`: the quoted work title in the purpose line when present
  (e.g. doctoral thesis title, 87% of rows), else the Finnish purpose line.
- `description`: the full purpose line.
- `funder_scheme`: Doctoral research grant (purpose 'vaitoskirjatyohon'),
  Postdoctoral research grant ('vaitoskirjan jalkeiseen tutkimukseen'),
  else Research grant. `funding_type` = fellowship for the first two, else research.
- `lead_investigator`: first named person; degree title stripped and the
  name split in Python (runbook 2.4.1). Co-grantees go to `investigators`.
  Organisation grantees have no lead unless written 'Org /Surname'.
- Dates: the site gives the grant year only; `start_year` = grant year,
  `start_date`/`end_date` NULL (no invented day/month).
- `amount`/`currency`: euros as printed ('32 000 e'); EUR.

**`funder_award_id` (2.1.1):** Wihuri publishes no grant number. Citing
works write the 8-digit application number (e.g. `00180353`; ~200 of the
330 acknowledgement shell rows for this funder), which the site never shows, so those shells cannot be collapsed. Synthetic
key `WIHURI-<year>-<sha1(year|grantee|purpose)[:10]>`; collisions raise in
the script. Zero existing shells use this form.

**Twin check:** `Wihuri Research Institute` (F4320330844) is a separate
institute the foundation runs; its own grants are not in this list's
science category beyond 8 operating grants to 'Wihurin tutkimuslaitos',
which stay under the foundation.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320334901`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 01gpyzq83, doi 10.13039/501100004022)

**Priority:** `526` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wihuri_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wihuri/wihuri_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants FROM openalex.awards.wihuri_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.wihuri_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.wihuri_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320334901 must resolve to exactly 1 row in `openalex.funders.funders`; the
assert fails the run otherwise.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320334901 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320334901;

## Step 2: Create Wihuri Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wihuri_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320334901  -- Jenny ja Antti Wihurin Rahasto
),

people AS (
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(from_json(people, 'array<struct<title:string,name:string,given_name:string,family_name:string>>'),
                   p -> p.family_name IS NOT NULL),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(lead_affiliation), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )) AS investigators
    FROM openalex.awards.wihuri_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.purpose), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    'EUR' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.funder_scheme IN ('Doctoral research grant', 'Postdoctoral research grant')
         THEN 'fellowship' ELSE 'research' END as funding_type,
    g.funder_scheme as funder_scheme,
    'wihuri_myonnot' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.grant_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- try_element_at: safe on empty arrays (2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.wihuri_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

## Shape check (2.1.1)
Every shipped `funder_award_id` must be the synthetic `WIHURI-<year>-<10 hex>`
key, and ids must be unique.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.wihuri_awards
WHERE NOT funder_award_id RLIKE '^WIHURI-(19|20)[0-9]{2}-[0-9a-f]{10}$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.wihuri_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wihuri_myonnot' AND priority = 526;

-- Priority 526: direct-from-funder ingest of Wihuri's own grant list.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    526 as priority
FROM openalex.awards.wihuri_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.wihuri_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.wihuri_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.wihuri_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.wihuri_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.wihuri_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.given_name) as has_pi_given
FROM openalex.awards.wihuri_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wihuri_myonnot'
GROUP BY provenance, priority;